https://github.com/karpathy/build-nanogpt/blob/master/fineweb.py

https://github.com/karpathy/build-nanogpt/blob/master/train_gpt2.py

# Datasets and Dataloaders

Using Dataset "FineWeb-Edu" From HuggingFace

In [ ]:
!pip install tiktoken
!pip install datasets
!pip install tqdm

In [ ]:
from datasets import load_dataset
import tiktoken
import os
import multiprocessing as mp
import numpy as np
from tqdm import tqdm


local_dir = "edu_fineweb10B"
remote_name = "sample-10BT"


# create the cache the local directory if it doesn't exist yet
DATA_CACHE_DIR = local_dir
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

# init the tokenizer
tokenizer = tiktoken.get_encoding("gpt2")
end_of_text = tokenizer._special_tokens['<|endoftext|>'] # end of text token

In [ ]:
# download the dataset
raw_ds = load_dataset("HuggingFaceFW/fineweb-edu", name=remote_name, split="train")

Right now when we go through the dataset it loads a DOCUMENT of TEXT at a time

And we'd like to do a few things to change that in pre-processing:

*   Tokenize all documents, and save them
*   Divide all the documents into "shards"

You can understand a shard as containing "shard_size" number of tokens

Basically concatenating however many documents necessary to meet that number




In [ ]:
num_processes = max(1, os.cpu_count() // 2)
shard_size = int(1e8) # 100M tokens per shard, total of 100 shards


# tokenizes a single document and returns a numpy array of uint16 tokens
def tokenize(doc):

    tokens = [end_of_text] # the special <|endoftext|> token delimits all documents
    tokens.extend(tokenizer.encode_ordinary(doc["text"]))
    tokens_np = np.array(tokens)
    return tokens_np.astype(np.uint16)


# save tokenized file in numpy binary file type
def write_datafile(filename, tokens_np):

    np.save(filename, tokens_np)


# tokenize all documents and write output shards, each of shard_size tokens (last shard has remainder)
with mp.Pool(num_processes) as pool:

    # preallocate buffer to hold current shard
    current_shard_tokens = np.empty((shard_size,), dtype=np.uint16)
    current_token_count = 0
    shard_index = 0
    # progress_bar = None

    for tokenized_document in pool.imap(tokenize, raw_ds, chunksize=16):

        # is there enough space in the current shard for the new tokens?
        if current_token_count + len(tokenized_document) < shard_size:

            # simply append tokens to current shard
            current_shard_tokens[current_token_count:current_token_count + len(tokenized_document)] = tokenized_document
            current_token_count += len(tokenized_document)

            # update progress bar
            # if progress_bar is None:
                # progress_bar = tqdm(total=shard_size, unit="tokens", desc=f"Shard {shard_index}")
            # progress_bar.update(len(tokenized_document))

        else:

            # write the current shard and start a new one
            split = "val" if shard_index == 0 else "train"
            filename = os.path.join(DATA_CACHE_DIR, f"edufineweb_{split}_{shard_index:06d}")

            # split the document into whatever fits in this shard; the remainder goes to next one
            remaining_space = shard_size - current_token_count
            # progress_bar.update(remaining_space)
            current_shard_tokens[current_token_count:current_token_count + remaining_space] = tokenized_document[:remaining_space]
            write_datafile(filename, current_shard_tokens)
            shard_index += 1
            print(f"Shard {shard_index} Complete")
            # progress_bar = None

            # populate the next shard with the leftovers of the current doc
            current_shard_tokens[0:len(tokenized_document) - remaining_space] = tokenized_document[remaining_space:]
            current_token_count = len(tokenized_document) - remaining_space

    # write any remaining tokens as the last shard
    if current_token_count != 0:
        split = "val" if shard_index == 0 else "train"
        filename = os.path.join(DATA_CACHE_DIR, f"edufineweb_{split}_{shard_index:06d}")
        write_datafile(filename, current_shard_tokens[:current_token_count])

In [ ]:
!zip -r /edu_fineweb10B.zip /edu_fineweb10B

In [ ]:
!unzip edu_fineweb10B.zip

creating pytorch dataset and dataloaders

we should iterate through a shard until we reach the end, then we bound over to the next shard and continue

this is because we can't load all shards at once into memory, that's way too large

https://medium.com/speechmatics/how-to-build-a-streaming-dataloader-with-pytorch-a66dd891d9dd

##  My Pytorch Iterable Dataset Implementation

It's mine so it might fail

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import IterableDataset, DataLoader


class My_FineWebEDU(IterableDataset):

    def __init__(self, data_root, batch_size, seq_len, split):
        super().__init__()

        self.data_root = data_root
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.split = split

        # get the shard filenames
        shards = os.listdir(data_root)
        shards = [shard for shard in shards if split in shard]
        shards = sorted(shards)
        shards = [os.path.join(data_root, shard) for shard in shards]
        self.shards = shards

        # Total number of shards
        self.total_shards = len(self.shards)


    def reset(self):

        self.current_shard_index = 0
        self.tokens = self.load_tokens(self.shards[self.current_shard_index])
        self.current_position = 0


    def load_tokens(self, file_path):

        tokens_np = np.load(file_path)
        tokens_np = tokens_np.astype(np.int32)
        tokens_torch = torch.tensor(tokens_np, dtype=torch.long)
        return tokens_torch


    def __iter__(self):
        self.reset()
        return self


    def __next__(self):

        # Move to the next shard if our current position goes out of index for the current shard
        if self.current_position + self.batch_size * self.seq_len + 1 > len(self.tokens):
            self.current_shard_index += 1

            if self.current_shard_index >= len(self.shards):
                raise StopIteration

            self.tokens = self.load_tokens(self.shards[self.current_shard_index])
            self.current_position = 0

        buffer = self.tokens[self.current_position:self.current_position + self.batch_size * self.seq_len + 1]
        inputs = buffer[:-1].clone().detach().long().view(self.batch_size, self.seq_len)
        targets = buffer[1:].clone().detach().long().view(self.batch_size, self.seq_len)
        self.current_position += self.batch_size * self.seq_len

        return inputs, targets

## Andrew's Implementation

No Pytorch Dataset Or DataLoader, therefore will need to rewrite training loop as well

In [ ]:
class Andrew_FineWebEDU:

    def __init__(self, data_root, batch_size, seq_len, process_rank, num_processes, split):
        self.B = batch_size
        self.T = seq_len
        self.process_rank = process_rank
        self.num_processes = num_processes

        # get the shard filenames
        shards = os.listdir(data_root)
        shards = [shard for shard in shards if split in shards]
        shards = sorted(shards)
        shards = [os.path.join(data_root, shard) for shard in shards]
        self.shards = shards

        self.reset()

    def reset(self):
        # state, init at shard zero
        self.current_shard = 0
        self.tokens = load_tokens(self.shards[self.current_shard])
        self.current_position = self.B * self.T * self.process_rank

    def next_batch(self):
        B, T = self.B, self.T

        buffer = self.tokens[self.current_position : self.current_position+B*T+1]
        inputs = (buffer[:-1]).view(B, T)
        targets = (buffer[1:]).view(B, T)

        # advance the position in the tensor
        self.current_position += B * T * self.num_processes

        # if loading the next batch would be out of bounds, advance to next shard
        if self.current_position + (B * T * self.num_processes + 1) > len(self.tokens):
            self.current_shard = (self.current_shard + 1) % len(self.shards)
            self.tokens = load_tokens(self.shards[self.current_shard])
            self.current_position = B * T * self.process_rank

        return inputs, targets

# GPT2 Architecture

Using Techniques For Speed:

*   flash attention
*   less assignment, more returns
*   nice numbers

Using Techniques For Performance:

*   special initialization




In [ ]:
from dataclasses import dataclass
from math import sqrt
import torch
import torch.nn as nn
import torch.nn.functional as F



@dataclass
class Our_GPT2Config:
    vocab_size: int = 50304 # number of tokens: 50,000 BPE merges + 256 bytes tokens + 1 <|endoftext|> token, but turned into "nice number" by + 47
    embedding_size: int = 768 # embedding dimension
    seq_len: int = 1024 # max sequence length
    num_layers: int = 12 # number of layers
    num_heads: int = 12 # number of heads
    end_of_text_token = tokenizer._special_tokens['<|endoftext|>'] # end of text token



# Exact Same Functionality as Char_GPT's Implementation of Attention, Just More Optimized!
class CasualSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.key = nn.Linear(config.embedding_size, config.embedding_size, bias=False)
        self.query = nn.Linear(config.embedding_size, config.embedding_size, bias=False)
        self.value = nn.Linear(config.embedding_size, config.embedding_size, bias=False)
        self.project = nn.Linear(config.embedding_size, config.embedding_size)
        self.project.flag = 1 # a "flag" for model initialization, feels like pytorch should have better implementation?

        self.num_heads = config.num_heads
        self.head_size = config.embedding_size // config.num_heads
        self.embedding_size = config.embedding_size

    def forward(self, x):
        B, T, C = x.size() # batch_size, seq_len, embedding_size (which sometimes is called model_size)

        # Get QKV
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        # Divide into heads (man I don't know how to reshape tensors)
        q = q.view(B, T, self.num_heads, self.head_size).transpose(1, 2) # (B, num_heads, T, head_size)
        k = k.view(B, T, self.num_heads, self.head_size).transpose(1, 2) # (B, num_heads, T, head_size)
        v = v.view(B, T, self.num_heads, self.head_size).transpose(1, 2) # (B, num_heads, T, head_size)

        # Flash Attention
        # out = F.scaled_dot_product_attention(q, k, v, is_causal=False)

        # Get all the head outputs together
        # out = out.transpose(1, 2).contiguous().view(B, T, C)

        # Projection
        # out = self.project(out)

        return self.project(F.scaled_dot_product_attention(q, k, v, is_causal=False).transpose(1, 2).contiguous().view(B, T, C))



class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.mlp_in = nn.Linear(config.embedding_size, 4 * config.embedding_size)
        self.GELU = nn.GELU() # can use tanh approximation, but no need to
        self.mlp_out = nn.Linear(4 * config.embedding_size, config.embedding_size)
        self.mlp_out.flag = 1 # a "flag" for model initialization, feels like pytorch should have better implementation?

    def forward(self, x):

        # x = self.mlp_in(x)
        # x = self.GELU(x)
        # x = self.mlp_out(x)

        return self.mlp_out(self.GELU(self.mlp_in(x)))



class GPT_Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(config.embedding_size)
        self.attn = CasualSelfAttention(config)
        self.layer_norm_2 = nn.LayerNorm(config.embedding_size)
        self.mlp = MLP(config)

    # norm & add
    def forward(self, x):
        x = x + self.attn(self.layer_norm_1(x))
        x = x + self.mlp(self.layer_norm_2(x))
        return x



class Our_GPT2(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            token_embeddings = nn.Embedding(config.vocab_size, config.embedding_size),
            positional_encodings = nn.Embedding(config.seq_len, config.embedding_size),
            blocks = nn.ModuleList([GPT_Block(config) for _ in range(config.num_layers)]),
            layer_norm_final = nn.LayerNorm(config.embedding_size),
            projection = nn.Linear(config.embedding_size, config.vocab_size)
        ))

        # weight sharing scheme, so these two "share" the same tensor, and apparently this just works better than them having separate values?
        self.transformer.token_embeddings.weight = self.transformer.projection.weight

        # init weights
        self.apply(self.init_weights)


    # initialize the weights of linear layers to a normal distribution
    def init_weights(self, module):

        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'flag'):
                std *= 1/sqrt(self.config.num_layers)
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


    def forward(self, inputs):

        # inputs is of shape (B, T)
        B, T = inputs.size()

        # forward the token and position embeddings
        tok_emb = self.transformer.token_embeddings(inputs) # (B,T,C)
        pos_emb = self.transformer.positional_encodings(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)

        # forward the blocks of the transformer
        for block in self.transformer.blocks:
            x = block(x)

        # forward the final layernorm and the classifier
        # x = self.transformer.layer_norm_final(x)
        # logits = self.transformer.projection(x) # (B, T, vocab_size)

        return self.transformer.projection(self.transformer.layer_norm_final(x))


        def generate(self, tokens, max_seq_len):

            # inputs is (B, T) array of indices in the current context
            while tokens.size(1) <  max_seq_len:

                with torch.no_grad():

                    # crop idx to the last seq_len tokens
                    context_window_tokens = tokens[:, -self.config.seq_len:]

                    # get the predictions
                    logits = model(context_window_tokens) # (B, T, vocab_size)

                    # focus only on the last time step
                    logits = logits[:, -1, :] # (B, vocab_size)

                    # apply softmax to get probabilities
                    probs = F.softmax(logits, dim=-1) # (B, vocab_size)

                    # only keep the top 50 probabilities
                    topk_probs, topk_indices = torch.topk(probs, 50, dim=-1) # (B, 50)

                    # select a token in topk
                    topk_token_index = torch.multinomial(topk_probs, num_samples=1) # (B, 1)

                    # gather the correspoding index in the vocabulary
                    vocab_token_index = torch.gather(topk_indices, dim=-1, index=topk_token_index) # (B, 1)

                    # if it's the <|endoftext|>' token, break out
                    if vocab_token_index == self.config.end_of_text:
                        break

                    # append sampled token index to the running sequence
                    tokens = torch.cat((tokens, vocab_token_index), dim=1) # (B, T+1)

            return input_tokens

# Training

Using Techniques For Speed:

*   mixed precision
*   torch.autocast
*   torch.compile

Using Techniques For Performance:

*   gradient clipping
*   learning rate scheduling (not implmeneted)
*   weight decay (not implmeneted)

In [ ]:
# cuda stuff
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device_type = "cuda" if device.startswith("cuda") else "cpu"
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.set_default_device(device)
torch.set_float32_matmul_precision('high')

## Using My Pytorch Iterable Dataset

In [ ]:
# hyperparameters
epochs = 1
eval_step_interval = 10000
learning_rate = 1e-3
batch_size = 16 # original paper was 64, way too large to store on GPU memory
seq_len = 1024

In [ ]:
# creating dataset and dataloaders
train_ds = My_FineWebEDU(local_dir, batch_size=batch_size, seq_len=seq_len, split="train")
val_ds = My_FineWebEDU(local_dir, batch_size=batch_size, seq_len=seq_len, split="val")
train_dataloader = DataLoader(train_ds, batch_size=None, generator=torch.Generator(device='cuda'))
val_dataloader = DataLoader(val_ds, batch_size=None, generator=torch.Generator(device='cuda'))

# creating model
model = Our_GPT2(Our_GPT2Config)
model.to(device)
model = torch.compile(model)
model.train()

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# create a Pytorch loss function
loss_fn = torch.nn.CrossEntropyLoss()

You might run out of memory here... even with a A100 GPU

There's no point to continue the training being honest... it's a huge model and I'm basically wasting compute power

https://discuss.pytorch.org/t/free-all-gpu-memory-used-in-between-runs/168202/2

In [ ]:
# training loop
steps = 0
for epoch in range(epochs):
    train_loss = 0.0

    for inputs, targets in train_dataloader:

        inputs, targets = inputs.to(device), targets.to(device)

        with torch.autocast(device_type=device_type, dtype=torch.bfloat16):
            logits = model(inputs)
            B,T,C = logits.shape
            loss = loss_fn(logits.contiguous().view(B*T,C), targets.contiguous().view(B*T))

        # evaluate the loss
        optimizer.zero_grad()
        loss.backward()
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.detach()
        steps += 1

        # every once in a while evaluate the loss on val dataset
        if steps % eval_step_interval == 0:

            model.eval()
            val_loss = 0.0
            val_len = 0

            with torch.no_grad():

                for inputs, targets in val_dataloader:

                    inputs, targets = inputs.to(device), targets.to(device)

                    with torch.autocast(device_type=device_type, dtype=torch.bfloat16):

                        logits = model(inputs)
                        B,T,C = logits.shape
                        loss = loss_fn(logits.contiguous().view(B*T,C), targets.contiguous().view(B*T))

                    val_loss += loss.detach()
                    val_len += 1

            model.train()

            print(f"step {steps}: train loss {train_loss/steps:.4f}, val loss {val_loss/val_len:.4f}")

    # every once in a while evaluate the loss on val dataset
    # if epoch % eval_interval == 0 or epoches - epoch == 1:

        # model.eval()
        # val_loss = 0.0
        # val_len = 0

        # with torch.no_grad():

            # for inputs, targets in val_dataloader:

                # inputs, targets = inputs.to(device), targets.to(device)

                # with torch.autocast(device_type=device_type, dtype=torch.bfloat16):

                    # logits = model(inputs)
                    # B,T,C = logits.shape
                    # loss = loss_fn(logits.contiguous().view(B*T,C), targets.contiguous().view(B*T))

                # val_loss += loss.detach()
                # val_len += 1

                # Free up memory
                # del inputs, targets, logits, loss
                # torch.cuda.empty_cache()

        # model.train()

        # print(f"step {steps}: train loss {train_loss/train_len:.4f}, val loss {val_loss/val_len:.4f}")

torch.save(model.state_dict(), f'model_state_dict_{steps}_steps.pth')

## Using Andrew's Implementation

In [ ]:
# hyperparameters
B = 16 # batch size, original paper was 64, way too large to store on GPU memory, can use 16 as well
T = 1024 # sequence length
max_steps = 152,584 # 19,073 steps is ~1 epoch (if it was batch size of 64, now multiplied by 4 since batch size was divided by 4), if data is 10B tokens and batch size 0.5M tokens
eval_interval = 250
process_rank = 0 # DPP stuff, not used
num_processes = 1 # DPP stuff, not used

total_batch_size = 524288 # 2**19, ~0.5M, in number of tokens
grad_accum_steps = total_batch_size // (B * T * num_processes) # gradient accumulation steps
learning_rate = 1e-3

In [ ]:
# creating dataloaders
train_loader = Andrew_FineWebEDU(B=B, T=T, process_rank=process_rank, num_processes=num_processes, split="train")
val_loader = Andrew_FineWebEDU(B=B, T=T, process_rank=process_rank, num_processes=num_processes, split="val")

# creating model
model = Our_GPT2(Our_GPT2Config)
model.to(device)
model = torch.compile(model)
model.train()

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# create a Pytorch loss function
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
for step in range(max_steps):

    last_step = (step == max_steps - 1)

    # once in a while evaluate our validation loss
    if step % 250 == 0 or last_step:

        model.eval()
        val_loader.reset()

        with torch.no_grad():

            val_loss_accum = 0.0
            val_loss_steps = 20

            for _ in range(val_loss_steps):

                inputs, targets = val_loader.next_batch()
                inputs, targets  = inputs.to(device), targets.to(device)

                with torch.autocast(device_type=device_type, dtype=torch.bfloat16):

                    logits = model(inputs, targets)
                    B,T,C = logits.shape
                    loss = loss_fn(logits.contiguous().view(B*T,C), targets.contiguous().view(B*T))

                loss = loss / val_loss_steps
                val_loss_accum += loss.detach()

        print(f"step {step:5d} | validation loss: {val_loss_accum.item():.4f}")


    # do one step of the optimization
    model.train()
    optimizer.zero_grad()
    loss_accum = 0.0

    for micro_step in range(grad_accum_steps):

        inputs, targets = train_loader.next_batch()
        inputs, targets  = inputs.to(device), targets.to(device)

        with torch.autocast(device_type=device_type, dtype=torch.bfloat16):

            logits = model(inputs, targets)
            B,T,C = logits.shape
            loss = loss_fn(logits.contiguous().view(B*T,C), targets.contiguous().view(B*T))

        loss = loss / grad_accum_steps
        loss_accum += loss.detach()
        loss.backward()

    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    print(f"step {step:5d} | train loss: {loss_accum.item():.6f} | norm: {norm:.4f}")


torch.save(model.state_dict(), f'model_state_dict_{max_steps}_steps.pth')

# Testing Model

In [ ]:
# load model
model = Our_GPT2(vocab_size, embedding_size, seq_len, num_heads, num_layers, dropout)
model.load_state_dict(torch.load(f'model_state_dict_{epochs}.pth'))
model = model.to(device)
model = torch.compile(model)
model.eval();

In [ ]:
# generate from the model
def generate_from_context(model, context_chars, seq_len, max_seq_len=2000):
    context_tokens = torch.tensor(tokenizer.encode(context_chars), dtype=torch.long, device=device).unsqueeze(0)
    return decode(model.generate(context_tokens, seq_len, max_seq_len)[0].tolist())

line = "GPT2! I wonder I torture myself."
output = generate_from_context(model, line, seq_len)
print(output)